# Notebook 04: Oil and gas pipeline corridor footprint

In [1]:
SECTOR = "Pipelines"
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

# I use the verified Notebook 03 exports without repeating the county spatial joins.
ROOT = Path(r"G:/My Drive/Projects/Large projects/2026TexasEnergyFootprint")
INPUT = ROOT / "05_Optimized_Data" / "GeoParquet"
OUTPUT = ROOT / "06_Energy_Footprint_Analysis" / SECTOR
OUTPUT.mkdir(parents=True, exist_ok=True)
ACRE_M2 = 4046.8564224
FT_PER_MILE = 5280.0
SQFT_PER_ACRE = 43560.0
SCENARIOS = ("low", "central", "high")

def read_layer(name):
    path = INPUT / f"{name}.parquet"
    if not path.is_file():
        raise FileNotFoundError(f"Missing Notebook 03 export: {path}")
    data = gpd.read_parquet(path)
    if data.crs is None:
        raise ValueError(f"{name}: missing CRS")
    if data.geometry.isna().any() or data.geometry.is_empty.any():
        raise ValueError(f"{name}: missing or empty geometries")
    return data

def county_key(df):
    if "county_geoid" not in df:
        raise KeyError("Notebook 03 county_geoid is required")
    s = df["county_geoid"].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(5)
    return s.where(s.str.fullmatch(r"48\d{3}"), pd.NA)

def save_results(detail, county, state, assumptions):
    # I save both geographic levels and verify that each scenario reconciles.
    for scenario in SCENARIOS:
        col = f"{scenario}_acres"
        if col not in detail or col not in county or col not in state:
            raise KeyError(f"Missing scenario output: {col}")
        a = float(detail[col].sum()); b = float(county[col].sum()); c = float(state[col].sum())
        if not np.isclose(a, c, rtol=1e-9, atol=0.01):
            raise AssertionError(f"Statewide {scenario} does not match facility detail")
        # I report unassigned county acreage separately rather than force a county allocation.
        unassigned = float(detail.loc[detail['county_geoid'].isna(), col].sum())
        if not np.isclose(b + unassigned, c, rtol=1e-9, atol=0.01):
            raise AssertionError(f"County plus unassigned acreage does not reconcile: {scenario}")
    detail.drop(columns=['geometry'], errors='ignore').to_csv(OUTPUT / 'facility_scenarios.csv', index=False)
    county.to_csv(OUTPUT / 'county_scenarios.csv', index=False)
    state.to_csv(OUTPUT / 'statewide_scenarios.csv', index=False)
    assumptions.to_csv(OUTPUT / 'scenario_assumptions.csv', index=False)
    print('Saved:', OUTPUT)
    display(state.round(2))
    display(county.head(10).round(2))

In [2]:
pipe=read_layer("Texas_Pipeline_County_Pieces_All_Statuses")
required=["pipeline_record_id","county_geoid","county_pipeline_miles","status_code","system_code"]
missing=[c for c in required if c not in pipe]
if missing:raise KeyError(f"Missing pipeline columns: {missing}")
pipe["county_geoid"]=county_key(pipe)
pipe["miles"]=pd.to_numeric(pipe["county_pipeline_miles"],errors="coerce")
assert pipe["miles"].notna().all() and pipe["miles"].ge(0).all()
assert np.isclose(pipe["miles"].sum(),362845.156062,atol=0.2)
print("All-status county miles:",pipe["miles"].sum())
print(pipe.groupby("status_code",dropna=False)["miles"].sum())
# I choose in-service records only; I do not use an unverified offshore proxy.
active=pipe.loc[pipe["status_code"].astype("string").str.strip().str.upper().eq("I")].copy()
assert len(active)>0
print("In-service county miles:",active["miles"].sum())

All-status county miles: 362845.15606194345
status_code
B     46596.409667
I    315010.008750
R      1238.737644
Name: miles, dtype: float64
In-service county miles: 315010.008750311


In [3]:
# I examine pipeline system types and operating statuses before
# deciding which records belong in the oil and gas footprint analysis.
# I use mileage rather than record counts alone because pipeline
# segments can vary substantially in length.

pipeline_system_audit = (
    pipe.groupby(
        ["system_code", "status_code"],
        dropna=False
    )
    .agg(
        county_pieces=("pipeline_record_id", "size"),
        source_records=("pipeline_record_id", "nunique"),
        pipeline_miles=("miles", "sum")
    )
    .reset_index()
    .sort_values(
        "pipeline_miles",
        ascending=False
    )
)

display(pipeline_system_audit)

print("\nTOTAL MILES BY SYSTEM TYPE")

display(
    pipe.groupby("system_code", dropna=False)
    .agg(
        county_pieces=("pipeline_record_id", "size"),
        pipeline_miles=("miles", "sum")
    )
    .sort_values("pipeline_miles", ascending=False)
)

print("\nIN-SERVICE MILES BY SYSTEM TYPE")

display(
    active.groupby("system_code", dropna=False)
    .agg(
        county_pieces=("pipeline_record_id", "size"),
        pipeline_miles=("miles", "sum")
    )
    .sort_values("pipeline_miles", ascending=False)
)

,system_code,status_code,county_pieces,source_records,pipeline_miles
6,G,I,339536,334298,170389.134688
22,T,I,54642,52813,50373.328757
20,Q,I,19957,18581,36617.548055
14,O,I,6236,5269,26497.936405
5,G,B,38507,37871,21571.079641
11,L,I,21518,21020,17175.121157
17,P,I,3551,3207,11586.301410
10,L,B,12514,12165,10368.542493
13,O,B,1900,1692,5681.229542
21,T,B,5346,5202,5098.969987



TOTAL MILES BY SYSTEM TYPE


,county_pieces,pipeline_miles
system_code,,
G,379243,192983.965717
T,60030,55546.981496
Q,21035,38887.014646
O,8151,32189.652305
L,34050,27581.922902
P,4220,12897.557786
K,478,2051.911104
Z,500,449.364787
C,43,189.004791



IN-SERVICE MILES BY SYSTEM TYPE


,county_pieces,pipeline_miles
system_code,,
G,339536,170389.134688
T,54642,50373.328757
Q,19957,36617.548055
O,6236,26497.936405
L,21518,17175.121157
P,3551,11586.301410
K,455,1969.712454
Z,298,251.140789
C,34,119.906869


In [4]:
# I inspect the columns retained in my pipeline datasets to determine
# whether commodity information was removed during earlier processing.
# I also check the available source-identification fields before
# deciding whether I need to return to the original RRC dataset.

for dataset_name in ["pipe", "active"]:
    if dataset_name in globals():
        dataset = globals()[dataset_name]

        print(f"\n{dataset_name.upper()}")
        print(f"Records: {len(dataset):,}")
        print("Columns:")
        print(dataset.columns.tolist())

print("\nPOTENTIAL ORIGINAL PIPELINE DATASETS")

for name, value in list(globals().items()):
    if (
        hasattr(value, "columns")
        and name not in ["pipe", "active"]
        and any(
            term in name.lower()
            for term in ["pipe", "rrc", "raw"]
        )
    ):
        print(f"\n{name}: {len(value):,} records")
        print(value.columns.tolist())


PIPE
Records: 507,807
Columns:
['pipeline_record_id', 'TPMS_ID', 'STATUS_CD', 'status_code', 'status_description', 'SYSTYPE', 'system_code', 'system_description', 'OPER_NM', 'SYS_NM', 'DIAMETER', 'CMDTY_DESC', 'ALBERS_MIL', 'county_geoid', 'county_name', 'county_pipeline_miles', 'geometry', 'miles']

ACTIVE
Records: 446,257
Columns:
['pipeline_record_id', 'TPMS_ID', 'STATUS_CD', 'status_code', 'status_description', 'SYSTYPE', 'system_code', 'system_description', 'OPER_NM', 'SYS_NM', 'DIAMETER', 'CMDTY_DESC', 'ALBERS_MIL', 'county_geoid', 'county_name', 'county_pipeline_miles', 'geometry', 'miles']

POTENTIAL ORIGINAL PIPELINE DATASETS

pipeline_system_audit: 27 records
['system_code', 'status_code', 'county_pieces', 'source_records', 'pipeline_miles']


In [5]:
# I examine the actual commodities transported by in-service
# pipelines to determine which records belong in my oil and gas
# infrastructure footprint analysis.
#
# I group by both system type and commodity because a pipeline's
# system classification does not necessarily establish its commodity.
# I retain missing commodity descriptions so I can assess uncertainty.

commodity_audit = (
    active
    .groupby(
        ["system_code", "system_description", "CMDTY_DESC"],
        dropna=False
    )
    .agg(
        county_pieces=("pipeline_record_id", "size"),
        pipeline_miles=("miles", "sum")
    )
    .reset_index()
    .sort_values(
        ["system_code", "pipeline_miles"],
        ascending=[True, False]
    )
)

display(commodity_audit)

# I calculate how much pipeline mileage lacks a commodity description
# because those records may require additional classification review.

missing_commodity = (
    active["CMDTY_DESC"].isna()
    | active["CMDTY_DESC"].astype("string").str.strip().eq("").fillna(False)
)

print(
    "\nIn-service pipeline miles with missing commodity:",
    f"{active.loc[missing_commodity, 'miles'].sum():,.2f}"
)

print(
    "Share of in-service mileage with missing commodity:",
    f"{active.loc[missing_commodity, 'miles'].sum() / active['miles'].sum() * 100:.2f}%"
)

,system_code,system_description,CMDTY_DESC,county_pieces,pipeline_miles
0,A,Crude Oil Offshore,CRUDE OIL OFFSHORE,30,29.878165
1,C,Anhydrous Ammonia,ANHYDROUS AMMONIA,34,119.906869
3,G,Gas Gathering,NATURAL GAS,335739,168706.380314
4,G,Gas Gathering,NATURAL GAS FWS,3732,1587.598127
2,G,Gas Gathering,0,65,95.156246
5,K,Carbon Dioxide,CARBON DIOXIDE,455,1969.712454
7,L,Crude Oil Gathering,CRUDE OIL,19737,16409.082492
6,L,Crude Oil Gathering,CRUDE FWS,1780,765.132785
8,L,Crude Oil Gathering,NATURAL GAS,1,0.905880
9,O,Crude Oil Transmission,CRUDE OIL,6236,26497.936405



In-service pipeline miles with missing commodity: 0.00
Share of in-service mileage with missing commodity: 0.00%


In [6]:
import numpy as np

# I classify each in-service pipeline using both its system type
# and commodity description. I keep uncertain categories separate
# rather than assuming every pipeline transports oil or natural gas.

pipeline_classified = active.copy()

commodity = (
    pipeline_classified["CMDTY_DESC"]
    .astype("string")
    .str.strip()
    .str.upper()
)

system = (
    pipeline_classified["system_code"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# I include natural gas, crude oil, and refined petroleum products.
# I also include FWS variants within the corresponding RRC
# gathering systems, but flag them for documentation.

confirmed = (
    (
        system.isin(["G", "T", "L", "O"])
        & commodity.isin([
            "NATURAL GAS",
            "NATURAL GAS FWS",
            "CRUDE OIL",
            "CRUDE FWS"
        ])
    )
    | (
        (system == "P")
        & (commodity == "REFINED LIQUID PRODUCT")
    )
)

# I retain HVL and other gas separately because the commodity
# descriptions do not identify the specific substances transported.

review = (
    (system == "Q")
    | (
        (system == "T")
        & commodity.eq("OTHER GAS")
    )
)

# I exclude explicitly non-petroleum and offshore systems from
# the primary onshore oil and gas footprint analysis.

excluded = system.isin(["A", "C", "K", "Z"]) | commodity.eq("HYDROGEN")

pipeline_classified["eligibility"] = np.select(
    [excluded, confirmed, review],
    ["Exclude", "Include", "Review"],
    default="Review"
)

# I summarize mileage by classification and check that no
# pipeline mileage was lost during this step.

eligibility_summary = (
    pipeline_classified
    .groupby("eligibility", as_index=False)
    .agg(
        county_pieces=("pipeline_record_id", "size"),
        pipeline_miles=("miles", "sum")
    )
)

eligibility_summary["share_pct"] = (
    eligibility_summary["pipeline_miles"]
    / pipeline_classified["miles"].sum() * 100
)

display(eligibility_summary)

assert np.isclose(
    eligibility_summary["pipeline_miles"].sum(),
    active["miles"].sum()
)

print("PASS: All in-service pipeline mileage accounted for.")

,eligibility,county_pieces,pipeline_miles,share_pct
0,Exclude,819,2372.816305,0.753251
1,Include,423834,274120.602834,87.019649
2,Review,21604,38516.589612,12.227100


PASS: All in-service pipeline mileage accounted for.


## ROW scenarios
I use 25, 37.5, and 50 feet as exploratory permanent ROW widths based on the original notebook’s review of National Academies Special Report 281 and FERC landowner guidance. Applying these transmission-oriented widths to gathering and other systems is an explicit proxy. Source leads: https://www.nationalacademies.org/read/11046/chapter/4 and https://www.ferc.gov/alternative-dispute-resolution/landowner-resources . **Gross ROW acreage = miles × 5,280 × ROW width (feet) ÷ 43,560.** 

In [7]:
assumptions=pd.DataFrame({"scenario":list(SCENARIOS),"row_width_feet":[25.0,37.5,50.0],"basis":["Exploratory lower permanent ROW width","Exploratory midpoint ROW width","Exploratory upper permanent ROW width"]})
for s,width in zip(SCENARIOS,assumptions["row_width_feet"]):active[f"{s}_acres"]=active["miles"]*FT_PER_MILE*width/SQFT_PER_ACRE
state=pd.DataFrame([{"sector":"Pipelines","status_included":"I","county_clipped_miles":active["miles"].sum(),"county_pieces":len(active),"source_records":active["pipeline_record_id"].nunique(),**{f"{s}_acres":active[f"{s}_acres"].sum() for s in SCENARIOS}}])
county=active.loc[active["county_geoid"].notna()].groupby("county_geoid",as_index=False).agg(county_pieces=("pipeline_record_id","size"),source_records=("pipeline_record_id","nunique"),miles=("miles","sum"),**{f"{s}_acres":(f"{s}_acres","sum") for s in SCENARIOS})
save_results(active,county,state,assumptions)
# I preserve all-status and in-service mileage separately for reproducibility.
pipe.groupby(["status_code","system_code"],dropna=False,as_index=False)["miles"].sum().to_csv(OUTPUT/"all_status_system_mileage_audit.csv",index=False)

Saved: G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\06_Energy_Footprint_Analysis\Pipelines


,sector,status_included,county_clipped_miles,county_pieces,source_records,low_acres,central_acres,high_acres
0,Pipelines,I,315010.01,446257,435922,954575.78,1431863.68,1909151.57


,county_geoid,county_pieces,source_records,miles,low_acres,central_acres,high_acres
0,48001,1536,1536,840.98,2548.43,3822.64,5096.85
1,48003,11721,11721,5152.20,15612.74,23419.11,31225.48
2,48005,1018,1018,690.25,2091.65,3137.48,4183.30
3,48007,223,223,212.01,642.45,963.67,1284.89
4,48009,242,242,369.03,1118.26,1677.39,2236.52
5,48011,12,12,95.79,290.26,435.39,580.52
6,48013,1658,1658,1263.33,3828.27,5742.41,7656.54
7,48015,716,716,779.14,2361.03,3541.54,4722.05
8,48017,24,24,152.16,461.08,691.62,922.16
9,48019,119,119,94.86,287.46,431.19,574.91


In [8]:
# I check how many Texas counties have in-service pipeline records.
print("Counties represented:", county["county_geoid"].nunique())
print("County rows:", len(county))
print("Missing county GEOIDs:", active["county_geoid"].isna().sum())

# I verify that county-level acreage reconciles with statewide acreage.
for scenario in SCENARIOS:
    difference = (
        state[f"{scenario}_acres"].iloc[0]
        - county[f"{scenario}_acres"].sum()
    )
    print(f"{scenario} acreage difference: {difference:,.4f}")

Counties represented: 253
County rows: 253
Missing county GEOIDs: 0
low acreage difference: 0.0000
central acreage difference: 0.0000
high acreage difference: 0.0000


In [9]:
# I retrieve the official Texas county names and GEOIDs from the U.S. Census.
import pandas as pd

census_url = (
    "https://www2.census.gov/geo/docs/reference/codes2020/"
    "national_county2020.txt"
)

county_reference = pd.read_csv(
    census_url,
    sep="|",
    dtype=str,
    encoding="utf-8-sig"
)

# I retain only Texas counties and standardize the identifiers.
county_reference = county_reference.loc[
    county_reference["STATEFP"] == "48",
    ["STATEFP", "COUNTYFP", "COUNTYNAME"]
].copy()

county_reference["county_geoid"] = (
    county_reference["STATEFP"] + county_reference["COUNTYFP"]
)

county_reference = county_reference.rename(
    columns={"COUNTYNAME": "county_name"}
)[["county_geoid", "county_name"]]

# I verify that the reference contains all 254 Texas counties.
assert len(county_reference) == 254
assert county_reference["county_geoid"].is_unique

# I standardize the GEOIDs before merging.
county["county_geoid"] = (
    county["county_geoid"]
    .astype("string")
    .str.zfill(5)
)

# I identify the county that has no in-service pipeline records.
missing_counties = county_reference.loc[
    ~county_reference["county_geoid"].isin(county["county_geoid"])
]

print("Counties missing from pipeline dataset:")
display(missing_counties)

# I retain all 254 counties, including those without pipeline records.
county_complete = county_reference.merge(
    county,
    on="county_geoid",
    how="left",
    validate="one_to_one"
)

numeric_columns = [
    "county_pieces",
    "source_records",
    "miles",
    "low_acres",
    "central_acres",
    "high_acres"
]

county_complete[numeric_columns] = (
    county_complete[numeric_columns].fillna(0)
)

# I sort the results alphabetically for data checking.
county_complete = (
    county_complete
    .sort_values("county_name")
    .reset_index(drop=True)
)

# I verify that adding the missing county does not change statewide acreage.
assert len(county_complete) == 254

for scenario in SCENARIOS:
    assert abs(
        county_complete[f"{scenario}_acres"].sum()
        - state[f"{scenario}_acres"].iloc[0]
    ) < 0.01

# I save the complete county-level results.
county_complete.to_csv(
    OUTPUT / "pipeline_county_254_with_names.csv",
    index=False
)

print("Total counties:", len(county_complete))
print("Counties with pipeline records:",
      (county_complete["county_pieces"] > 0).sum())
print("Counties without pipeline records:",
      (county_complete["county_pieces"] == 0).sum())

display(county_complete.head(10))

Counties missing from pipeline dataset:


,county_geoid,county_name
2586,48125,Dickens County


Total counties: 254
Counties with pipeline records: 253
Counties without pipeline records: 1


,county_geoid,county_name,county_pieces,source_records,miles,low_acres,central_acres,high_acres
0,48001,Anderson County,1536.0,1536.0,840.980777,2548.426596,3822.639895,5096.853193
1,48003,Andrews County,11721.0,11721.0,5152.204590,15612.741180,23419.111771,31225.482361
2,48005,Angelina County,1018.0,1018.0,690.245217,2091.652172,3137.478258,4183.304344
3,48007,Aransas County,223.0,223.0,212.007276,642.446292,963.669439,1284.892585
4,48009,Archer County,242.0,242.0,369.025502,1118.259097,1677.388646,2236.518194
5,48011,Armstrong County,12.0,12.0,95.785438,290.258903,435.388355,580.517806
6,48013,Atascosa County,1658.0,1658.0,1263.329700,3828.271819,5742.407729,7656.543638
7,48015,Austin County,716.0,716.0,779.138634,2361.026162,3541.539244,4722.052325
8,48017,Bailey County,24.0,24.0,152.155725,461.077954,691.616930,922.155907
9,48019,Bandera County,119.0,119.0,94.860773,287.456889,431.185333,574.913777


In [10]:
# I calculate the gross right-of-way footprint of in-service pipelines
# under three exploratory width scenarios.

from pathlib import Path
import pandas as pd
import numpy as np

OUTPUT = Path(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)

# I document the assumptions used to estimate pipeline right-of-way acreage.
assumptions = pd.DataFrame({
    "scenario": ["low", "central", "high"],
    "row_width_feet": [25.0, 37.5, 50.0],
    "basis": [
        "Exploratory lower permanent ROW width",
        "Exploratory midpoint ROW width",
        "Exploratory upper permanent ROW width"
    ]
})

# I use the in-service pipeline subset and preserve the original data.
active = pipe.loc[
    pipe["status_code"].astype("string").str.strip().eq("I")
].copy()

# I verify that the mileage and county identifiers are available.
required_columns = [
    "pipeline_record_id",
    "county_geoid",
    "miles",
    "status_code"
]

missing_columns = [
    col for col in required_columns if col not in active.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"
assert active["county_geoid"].notna().all(), (
    "Some in-service pipeline pieces have no county GEOID."
)
assert active["miles"].notna().all(), (
    "Some in-service pipeline pieces have missing mileage."
)
assert (active["miles"] >= 0).all(), (
    "Some in-service pipeline pieces have negative mileage."
)

# I calculate gross right-of-way acreage for each scenario.
for scenario, width in zip(
    assumptions["scenario"],
    assumptions["row_width_feet"]
):
    active[f"{scenario}_acres"] = (
        active["miles"]
        * FT_PER_MILE
        * width
        / SQFT_PER_ACRE
    )

# I calculate statewide totals using the county-clipped pipeline pieces.
state = pd.DataFrame([{
    "sector": "Pipelines",
    "status_included": "I",
    "county_clipped_miles": active["miles"].sum(),
    "county_pieces": len(active),
    "source_records": active["pipeline_record_id"].nunique(),
    **{
        f"{scenario}_acres": active[f"{scenario}_acres"].sum()
        for scenario in assumptions["scenario"]
    }
}])

# I calculate county-level totals before adding counties with no records.
county = (
    active.groupby("county_geoid", as_index=False)
    .agg(
        county_pieces=("pipeline_record_id", "size"),
        source_records=("pipeline_record_id", "nunique"),
        miles=("miles", "sum"),
        **{
            f"{scenario}_acres": (f"{scenario}_acres", "sum")
            for scenario in assumptions["scenario"]
        }
    )
)

# I retrieve the official 2020 Census county reference for Texas.
census_url = (
    "https://www2.census.gov/geo/docs/reference/codes2020/"
    "national_county2020.txt"
)

county_reference = pd.read_csv(
    census_url,
    sep="|",
    dtype=str,
    encoding="utf-8-sig"
)

county_reference = county_reference.loc[
    county_reference["STATEFP"] == "48",
    ["STATEFP", "COUNTYFP", "COUNTYNAME"]
].copy()

county_reference["county_geoid"] = (
    county_reference["STATEFP"]
    + county_reference["COUNTYFP"]
)

county_reference = county_reference.rename(
    columns={"COUNTYNAME": "county_name"}
)[["county_geoid", "county_name"]]

# I verify that the reference contains exactly 254 unique Texas counties.
assert len(county_reference) == 254
assert county_reference["county_geoid"].is_unique

# I standardize county identifiers to preserve leading zeros.
county["county_geoid"] = (
    county["county_geoid"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

# I verify that every county identifier matches the Texas county reference.
unrecognized = set(county["county_geoid"]) - set(
    county_reference["county_geoid"]
)

assert not unrecognized, (
    f"Unrecognized county GEOIDs: {sorted(unrecognized)}"
)

# I identify counties without in-service pipeline records.
missing_counties = county_reference.loc[
    ~county_reference["county_geoid"].isin(county["county_geoid"])
].copy()

print("Counties without in-service pipeline records:")
print(missing_counties.to_string(index=False))

# I add the missing counties without changing the measured pipeline mileage.
county_complete = county_reference.merge(
    county,
    on="county_geoid",
    how="left",
    validate="one_to_one"
)

numeric_columns = [
    "county_pieces",
    "source_records",
    "miles",
    "low_acres",
    "central_acres",
    "high_acres"
]

county_complete[numeric_columns] = (
    county_complete[numeric_columns].fillna(0)
)

# I retain integer counts for pipeline pieces and source records.
county_complete["county_pieces"] = (
    county_complete["county_pieces"].astype(int)
)
county_complete["source_records"] = (
    county_complete["source_records"].astype(int)
)

# I sort the complete county results alphabetically.
county_complete = (
    county_complete
    .sort_values("county_name")
    .reset_index(drop=True)
)

# I verify the statewide totals against the sum of all 254 counties.
assert len(county_complete) == 254
assert county_complete["county_geoid"].is_unique

assert np.isclose(
    county_complete["miles"].sum(),
    state["county_clipped_miles"].iloc[0],
    rtol=0,
    atol=0.01
)

for scenario in assumptions["scenario"]:
    statewide_acres = state[f"{scenario}_acres"].iloc[0]
    county_acres = county_complete[f"{scenario}_acres"].sum()

    assert np.isclose(
        statewide_acres,
        county_acres,
        rtol=0,
        atol=0.01
    ), f"{scenario} acreage does not reconcile."

# I save the complete facility-level, county-level, statewide,
# and scenario-assumption outputs using the existing notebook helper.
save_results(
    active,
    county_complete,
    state,
    assumptions
)

# I save an explicitly named county CSV for the data checker.
county_complete.to_csv(
    OUTPUT / "pipeline_county_254_with_names.csv",
    index=False
)

# I preserve all pipeline statuses for a separate mileage audit.
pipe.groupby(
    ["status_code", "system_code"],
    dropna=False,
    as_index=False
)["miles"].sum().to_csv(
    OUTPUT / "all_status_system_mileage_audit.csv",
    index=False
)

# I report the final validation results.
print("\nFINAL PIPELINE VALIDATION")
print("Texas counties:", len(county_complete))
print(
    "Counties with in-service records:",
    (county_complete["county_pieces"] > 0).sum()
)
print(
    "Counties without in-service records:",
    (county_complete["county_pieces"] == 0).sum()
)
print("In-service pipeline miles:", f"{active['miles'].sum():,.2f}")

for scenario in assumptions["scenario"]:
    print(
        f"{scenario.capitalize()} acreage:",
        f"{state[f'{scenario}_acres'].iloc[0]:,.2f}"
    )

print("\nPASS: Statewide and county acreage calculations reconcile.")

display(state)
display(county_complete)

Counties without in-service pipeline records:
county_geoid    county_name
       48125 Dickens County
Saved: G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\06_Energy_Footprint_Analysis\Pipelines


,sector,status_included,county_clipped_miles,county_pieces,source_records,low_acres,central_acres,high_acres
0,Pipelines,I,315010.01,446257,435922,954575.78,1431863.68,1909151.57


,county_geoid,county_name,county_pieces,source_records,miles,low_acres,central_acres,high_acres
0,48001,Anderson County,1536,1536,840.98,2548.43,3822.64,5096.85
1,48003,Andrews County,11721,11721,5152.20,15612.74,23419.11,31225.48
2,48005,Angelina County,1018,1018,690.25,2091.65,3137.48,4183.30
3,48007,Aransas County,223,223,212.01,642.45,963.67,1284.89
4,48009,Archer County,242,242,369.03,1118.26,1677.39,2236.52
5,48011,Armstrong County,12,12,95.79,290.26,435.39,580.52
6,48013,Atascosa County,1658,1658,1263.33,3828.27,5742.41,7656.54
7,48015,Austin County,716,716,779.14,2361.03,3541.54,4722.05
8,48017,Bailey County,24,24,152.16,461.08,691.62,922.16
9,48019,Bandera County,119,119,94.86,287.46,431.19,574.91



FINAL PIPELINE VALIDATION
Texas counties: 254
Counties with in-service records: 253
Counties without in-service records: 1
In-service pipeline miles: 315,010.01
Low acreage: 954,575.78
Central acreage: 1,431,863.68
High acreage: 1,909,151.57

PASS: Statewide and county acreage calculations reconcile.


,sector,status_included,county_clipped_miles,county_pieces,source_records,low_acres,central_acres,high_acres
0,Pipelines,I,315010.00875,446257,435922,954575.784092,1.431864e+06,1.909152e+06


,county_geoid,county_name,county_pieces,source_records,miles,low_acres,central_acres,high_acres
0,48001,Anderson County,1536,1536,840.980777,2548.426596,3822.639895,5096.853193
1,48003,Andrews County,11721,11721,5152.204590,15612.741180,23419.111771,31225.482361
2,48005,Angelina County,1018,1018,690.245217,2091.652172,3137.478258,4183.304344
3,48007,Aransas County,223,223,212.007276,642.446292,963.669439,1284.892585
4,48009,Archer County,242,242,369.025502,1118.259097,1677.388646,2236.518194
...,...,...,...,...,...,...,...,...
249,48499,Wood County,562,562,397.810780,1205.487213,1808.230819,2410.974426
250,48501,Yoakum County,1247,1247,1562.427597,4734.629082,7101.943623,9469.258164
251,48503,Young County,2369,2369,962.366235,2916.261317,4374.391976,5832.522634
252,48505,Zapata County,3857,3857,2196.836185,6657.079350,9985.619025,13314.158700


In [11]:
# I calculate an additional pipeline land-footprint scenario to account
# for differences in right-of-way (ROW) widths across pipeline systems.
#
# In my original analysis, I applied the same three ROW width scenarios
# to all in-service pipelines: 25 feet (low), 37.5 feet (central), and
# 50 feet (high). However, transmission pipelines generally require
# wider corridors than gathering and other smaller pipeline systems.
# Applying the same width to every pipeline may therefore overlook
# differences in their land-use requirements.
#
# To address this limitation, I create a mixed-width scenario that
# applies the high ROW assumption (50 feet) to transmission pipelines
# identified by RRC system code "T" and the central ROW assumption
# (37.5 feet) to all other in-service pipeline systems.
#
# I use the same pipeline records included in my original acreage
# analysis and do not modify the existing low, central, or high results.
# This allows me to compare the mixed-width scenario directly with
# the original estimates and evaluate how using different ROW widths
# affects the total pipeline land footprint.
#
# I calculate the new acreage for each pipeline segment, aggregate
# the results statewide and across all 254 Texas counties, and
# verify that the county totals reconcile with the statewide total.
#
# I save the mixed-width results in separate CSV files for further
# analysis and comparison with other energy infrastructure.
#
# These ROW widths remain analytical assumptions rather than
# measured corridor widths for individual pipelines. The resulting
# acreage represents estimated ROW corridor area, not necessarily
# the area of permanent physical land disturbance.

In [12]:
import numpy as np
import pandas as pd
from IPython.display import display

# I calculate an additional pipeline ROW scenario without changing
# my existing low, central, or high scenario results.
#
# I apply the high ROW width (50 feet) to transmission pipelines
# identified by RRC system code T.
#
# I apply the central ROW width (37.5 feet) to all other in-service
# pipeline systems, including gathering and other pipeline types.
#
# I calculate this as a separate mixed-width scenario to evaluate
# Tony's suggestion. The widths remain exploratory assumptions.
#
# I use the same in-service pipeline population as my existing
# acreage analysis so the new results are directly comparable.
# Commodity eligibility will be reviewed separately.

CENTRAL_ROW_FEET = 37.5
TRANSMISSION_ROW_FEET = 50.0

# I preserve the original active dataset and its existing results.
pipeline_mixed = active.copy()

# I verify that the required columns exist.
required = [
    "pipeline_record_id",
    "county_geoid",
    "system_code",
    "miles",
    "central_acres",
    "high_acres"
]

missing = [
    col for col in required
    if col not in pipeline_mixed.columns
]

assert not missing, f"Missing columns: {missing}"

# I identify transmission pipelines using system code T.
pipeline_mixed["is_transmission"] = (
    pipeline_mixed["system_code"]
    .astype("string")
    .str.strip()
    .str.upper()
    .eq("T")
    .fillna(False)
)

# I assign 50 feet to transmission pipelines and 37.5 feet
# to every other in-service pipeline system.
pipeline_mixed["selected_row_width_feet"] = np.where(
    pipeline_mixed["is_transmission"],
    TRANSMISSION_ROW_FEET,
    CENTRAL_ROW_FEET
)

# I calculate the new acreage using the selected ROW width.
pipeline_mixed["mixed_row_acres"] = (
    pipeline_mixed["miles"]
    * FT_PER_MILE
    * pipeline_mixed["selected_row_width_feet"]
    / SQFT_PER_ACRE
)

# I verify that the mixed scenario uses the existing high
# acreage for transmission and central acreage for other systems.
expected_acres = np.where(
    pipeline_mixed["is_transmission"],
    pipeline_mixed["high_acres"],
    pipeline_mixed["central_acres"]
)

assert np.allclose(
    pipeline_mixed["mixed_row_acres"],
    expected_acres,
    rtol=0,
    atol=0.01
)

# I summarize the mileage and acreage by ROW assumption.
pipeline_mixed["row_category"] = np.where(
    pipeline_mixed["is_transmission"],
    "Transmission - high ROW",
    "Other pipelines - central ROW"
)

pipeline_mixed_summary = (
    pipeline_mixed
    .groupby(
        ["row_category", "selected_row_width_feet"],
        as_index=False
    )
    .agg(
        county_pieces=("pipeline_record_id", "size"),
        pipeline_miles=("miles", "sum"),
        mixed_row_acres=("mixed_row_acres", "sum")
    )
)

# I calculate the mixed-width footprint for each Texas county.
pipeline_mixed_county = (
    pipeline_mixed
    .groupby("county_geoid", as_index=False)
    .agg(
        pipeline_miles=("miles", "sum"),
        transmission_miles=(
            "miles",
            lambda x: x[
                pipeline_mixed.loc[x.index, "is_transmission"]
            ].sum()
        ),
        mixed_row_acres=("mixed_row_acres", "sum")
    )
)

# I retain all 254 Texas counties, including counties
# without in-service pipeline records.
pipeline_mixed_county_254 = county_reference.merge(
    pipeline_mixed_county,
    on="county_geoid",
    how="left",
    validate="one_to_one"
)

numeric_columns = [
    "pipeline_miles",
    "transmission_miles",
    "mixed_row_acres"
]

pipeline_mixed_county_254[numeric_columns] = (
    pipeline_mixed_county_254[numeric_columns].fillna(0)
)

# I add the existing central and high estimates for comparison.
pipeline_mixed_county_254 = (
    pipeline_mixed_county_254.merge(
        county_complete[
            ["county_geoid", "central_acres", "high_acres"]
        ],
        on="county_geoid",
        how="left",
        validate="one_to_one"
    )
)

pipeline_mixed_county_254["additional_acres_vs_central"] = (
    pipeline_mixed_county_254["mixed_row_acres"]
    - pipeline_mixed_county_254["central_acres"]
)

pipeline_mixed_county_254 = (
    pipeline_mixed_county_254
    .sort_values("county_name")
    .reset_index(drop=True)
)

# I create a statewide comparison of the existing central,
# new mixed-width, and existing high scenarios.
pipeline_mixed_state = pd.DataFrame([{
    "sector": "Pipelines",
    "pipeline_miles": pipeline_mixed["miles"].sum(),
    "central_all_acres": pipeline_mixed["central_acres"].sum(),
    "mixed_row_acres": pipeline_mixed["mixed_row_acres"].sum(),
    "high_all_acres": pipeline_mixed["high_acres"].sum()
}])

pipeline_mixed_state["additional_acres_vs_central"] = (
    pipeline_mixed_state["mixed_row_acres"]
    - pipeline_mixed_state["central_all_acres"]
)

# I verify that county totals reconcile with statewide results.
assert len(pipeline_mixed_county_254) == 254

assert np.isclose(
    pipeline_mixed_county_254["mixed_row_acres"].sum(),
    pipeline_mixed_state["mixed_row_acres"].iloc[0],
    rtol=0,
    atol=0.01
)

assert np.isclose(
    pipeline_mixed_county_254["pipeline_miles"].sum(),
    pipeline_mixed_state["pipeline_miles"].iloc[0],
    rtol=0,
    atol=0.01
)

# I verify that the mixed result falls between the existing
# central and high estimates.
assert (
    pipeline_mixed_state["central_all_acres"].iloc[0]
    <= pipeline_mixed_state["mixed_row_acres"].iloc[0]
    <= pipeline_mixed_state["high_all_acres"].iloc[0]
)

# I save the mixed scenario separately so my original
# pipeline scenario outputs remain unchanged.
pipeline_mixed_summary.to_csv(
    OUTPUT / "pipeline_mixed_row_by_type.csv",
    index=False
)

pipeline_mixed_state.to_csv(
    OUTPUT / "pipeline_mixed_row_statewide.csv",
    index=False
)

pipeline_mixed_county_254.to_csv(
    OUTPUT / "pipeline_mixed_row_county_254.csv",
    index=False
)

print("PASS: Mixed ROW scenario calculated.")
print("PASS: All 254 counties included.")
print("PASS: County and statewide totals reconciled.")

print("\nROW WIDTH BY PIPELINE CATEGORY")
display(pipeline_mixed_summary.round(2))

print("\nSTATEWIDE COMPARISON")
display(pipeline_mixed_state.round(2))

print("\nCOUNTY-BY-COUNTY COMPARISON")
display(pipeline_mixed_county_254.round(2))

print("\nSaved three new CSV files to:", OUTPUT)

PASS: Mixed ROW scenario calculated.
PASS: All 254 counties included.
PASS: County and statewide totals reconciled.

ROW WIDTH BY PIPELINE CATEGORY


,row_category,selected_row_width_feet,county_pieces,pipeline_miles,mixed_row_acres
0,Other pipelines - central ROW,37.5,391615,264636.68,1202894.0
1,Transmission - high ROW,50.0,54642,50373.33,305292.9



STATEWIDE COMPARISON


,sector,pipeline_miles,central_all_acres,mixed_row_acres,high_all_acres,additional_acres_vs_central
0,Pipelines,315010.01,1431863.68,1508186.9,1909151.57,76323.23



COUNTY-BY-COUNTY COMPARISON


,county_geoid,county_name,pipeline_miles,transmission_miles,mixed_row_acres,central_acres,high_acres,additional_acres_vs_central
0,48001,Anderson County,840.98,429.07,4472.74,3822.64,5096.85,650.10
1,48003,Andrews County,5152.20,243.69,23788.34,23419.11,31225.48,369.23
2,48005,Angelina County,690.25,227.40,3482.03,3137.48,4183.30,344.55
3,48007,Aransas County,212.01,29.93,1009.03,963.67,1284.89,45.36
4,48009,Archer County,369.03,51.41,1755.29,1677.39,2236.52,77.90
...,...,...,...,...,...,...,...,...
249,48499,Wood County,397.81,135.81,2014.00,1808.23,2410.97,205.77
250,48501,Yoakum County,1562.43,114.88,7276.01,7101.94,9469.26,174.07
251,48503,Young County,962.37,34.09,4426.04,4374.39,5832.52,51.65
252,48505,Zapata County,2196.84,247.39,10360.45,9985.62,13314.16,374.83



Saved three new CSV files to: G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\06_Energy_Footprint_Analysis\Pipelines


In [13]:
# I apply my earlier commodity eligibility classification
# to the mixed-width pipeline scenario.
#
# I preserve the existing 50-foot assumption for system T
# and 37.5 feet for other systems so I can isolate the
# effect of commodity exclusions.

classified = pipeline_classified.copy()

classified["system_code"] = (
    classified["system_code"]
    .astype("string")
    .str.strip()
    .str.upper()
)

classified["row_width_ft"] = np.where(
    classified["system_code"].eq("T"),
    50.0,
    37.5
)

classified["revised_acres"] = (
    classified["miles"]
    * classified["row_width_ft"]
    / 8.25
)

# I summarize confirmed, excluded, and uncertain pipelines
# before selecting the final analysis population.

eligibility_check = (
    classified.groupby("eligibility", dropna=False)
    .agg(
        records=("pipeline_record_id", "size"),
        miles=("miles", "sum"),
        acres=("revised_acres", "sum")
    )
    .round(2)
)

display(eligibility_check)

print(
    "\nOriginal mixed-width acreage:",
    f"{pipeline_mixed_state['mixed_row_acres'].iloc[0]:,.2f}"
)

,records,miles,acres
eligibility,,,
Exclude,819,2372.82,10788.83
Include,423834,274120.60,1319589.51
Review,21604,38516.59,177808.57



Original mixed-width acreage: 1,508,186.90


In [14]:
# I examine the pipeline types in the Review category
# before deciding which records should remain in my analysis.

review_breakdown = (
    pipeline_classified.loc[
        pipeline_classified["eligibility"] == "Review"
    ]
    .groupby(
        ["system_code", "system_description", "CMDTY_DESC"],
        dropna=False
    )
    .agg(
        records=("pipeline_record_id", "size"),
        miles=("miles", "sum")
    )
    .sort_values("miles", ascending=False)
    .round(2)
)

display(review_breakdown)

records  \
system_code system_description     CMDTY_DESC                              
Q           Highly Volatile Liquid HIGHLY VOLATILE LIQUID          18525   
                                   HIGHLY VOLATILE LIQUID (HVL)     1432   
T           Gas Transmission       OTHER GAS                        1582   
G           Gas Gathering          0                                  65   

                                                                    miles  
system_code system_description     CMDTY_DESC                              
Q           Highly Volatile Liquid HIGHLY VOLATILE LIQUID        31746.63  
                                   HIGHLY VOLATILE LIQUID (HVL)   4870.92  
T           Gas Transmission       OTHER GAS                      1803.89  
G           Gas Gathering          0                                95.16

In [15]:
# I retain all records classified as Include or Review,
# including HVL, OTHER GAS, and unknown commodity records.
#
# I exclude records classified as Exclude and preserve
# the original mixed-width ROW assumptions.

pipeline_final = classified.loc[
    classified["eligibility"].isin(["Include", "Review"])
].copy()

print("Final included records:", len(pipeline_final))
print(
    "Final included miles:",
    f"{pipeline_final['miles'].sum():,.2f}"
)
print(
    "Final modeled ROW acres:",
    f"{pipeline_final['revised_acres'].sum():,.2f}"
)

print("\nIncluded commodity descriptions:")
display(
    pipeline_final.groupby("CMDTY_DESC", dropna=False)
    .agg(
        records=("pipeline_record_id", "size"),
        miles=("miles", "sum"),
        acres=("revised_acres", "sum")
    )
    .sort_values("miles", ascending=False)
    .round(2)
)

Final included records: 445438
Final included miles: 312,637.19
Final modeled ROW acres: 1,497,398.07

Included commodity descriptions:


,records,miles,acres
CMDTY_DESC,,,
NATURAL GAS,388798,217274.55,1061198.36
CRUDE OIL,25973,42907.02,195031.90
HIGHLY VOLATILE LIQUID,18525,31746.63,144302.85
REFINED LIQUID PRODUCT,3551,11586.30,52665.01
HIGHLY VOLATILE LIQUID (HVL),1432,4870.92,22140.55
OTHER GAS,1582,1803.89,10932.64
NATURAL GAS FWS,3732,1587.60,7216.36
CRUDE FWS,1780,765.13,3477.88
0,65,95.16,432.53


In [16]:
# I inspect the geographic identifiers in my final
# pipeline records before creating county-level outputs.

print("Final pipeline columns:")
print(pipeline_final.columns.tolist())

print("\nCounty assignment fields:")
for col in ["county_geoid", "county_name"]:
    if col in pipeline_final.columns:
        print(
            col,
            "missing:",
            pipeline_final[col].isna().sum(),
            "unique:",
            pipeline_final[col].nunique()
        )

print("\nStatewide acreage:")
print(f"{pipeline_final['revised_acres'].sum():,.2f}")

print("\nStatewide mileage:")
print(f"{pipeline_final['miles'].sum():,.2f}")

Final pipeline columns:
['pipeline_record_id', 'TPMS_ID', 'STATUS_CD', 'status_code', 'status_description', 'SYSTYPE', 'system_code', 'system_description', 'OPER_NM', 'SYS_NM', 'DIAMETER', 'CMDTY_DESC', 'ALBERS_MIL', 'county_geoid', 'county_name', 'county_pipeline_miles', 'geometry', 'miles', 'eligibility', 'row_width_ft', 'revised_acres']

County assignment fields:
county_geoid missing: 0 unique: 253
county_name missing: 0 unique: 253

Statewide acreage:
1,497,398.07

Statewide mileage:
312,637.19


In [17]:
from pathlib import Path
import pandas as pd
import numpy as np

# I use the validated 254-county reference from my earlier
# pipeline analysis to preserve counties without eligible pipelines.

pipeline_folder = (
    Path(r"G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint")
    / "06_Energy_Footprint_Analysis"
    / "Pipelines"
)

reference = pd.read_csv(
    pipeline_folder / "pipeline_mixed_row_county_254.csv",
    dtype={"county_geoid": "string"}
)

county_reference = reference[
    ["county_geoid", "county_name"]
].drop_duplicates(subset="county_geoid")

# I summarize the final eligible pipeline records by county,
# preserving the original geographic allocation.

pipeline_final["county_geoid"] = (
    pipeline_final["county_geoid"]
    .astype("string")
    .str.zfill(5)
)

county_summary = (
    pipeline_final
    .groupby("county_geoid", as_index=False)
    .agg(
        pipeline_records=("pipeline_record_id", "size"),
        pipeline_miles=("miles", "sum"),
        mixed_row_acres=("revised_acres", "sum")
    )
)

# I merge the results with all 254 Texas counties so that
# counties without eligible pipelines receive zero acreage.

pipeline_county_final = county_reference.merge(
    county_summary,
    on="county_geoid",
    how="left",
    validate="one_to_one"
)

numeric_columns = [
    "pipeline_records",
    "pipeline_miles",
    "mixed_row_acres"
]

pipeline_county_final[numeric_columns] = (
    pipeline_county_final[numeric_columns].fillna(0)
)

pipeline_county_final["pipeline_records"] = (
    pipeline_county_final["pipeline_records"].astype(int)
)

# I verify that the county aggregation preserves every
# eligible pipeline record and the complete statewide acreage.

print("Texas counties:", len(pipeline_county_final))

print(
    "County pipeline records:",
    f"{pipeline_county_final['pipeline_records'].sum():,}"
)

print(
    "County pipeline miles:",
    f"{pipeline_county_final['pipeline_miles'].sum():,.2f}"
)

print(
    "County pipeline acres:",
    f"{pipeline_county_final['mixed_row_acres'].sum():,.2f}"
)

print(
    "Statewide pipeline acres:",
    f"{pipeline_final['revised_acres'].sum():,.2f}"
)

assert len(pipeline_county_final) == 254

assert np.isclose(
    pipeline_county_final["mixed_row_acres"].sum(),
    pipeline_final["revised_acres"].sum(),
    atol=0.01,
    rtol=0
)

assert (
    pipeline_county_final["pipeline_records"].sum()
    == len(pipeline_final)
)

print("\nCounties with zero eligible pipeline acreage:")
display(
    pipeline_county_final.loc[
        pipeline_county_final["mixed_row_acres"] == 0,
        ["county_geoid", "county_name"]
    ]
)

print("\nCounty validation passed.")

Texas counties: 254
County pipeline records: 445,438
County pipeline miles: 312,637.19
County pipeline acres: 1,497,398.07
Statewide pipeline acres: 1,497,398.07

Counties with zero eligible pipeline acreage:


,county_geoid,county_name
62,48125,Dickens County



County validation passed.


In [18]:
# I save the revised pipeline results under new filenames
# so I preserve the original analysis for comparison.

statewide_revised = pd.DataFrame([{
    "pipeline_records": len(pipeline_final),
    "pipeline_miles": pipeline_final["miles"].sum(),
    "mixed_row_acres": pipeline_final["revised_acres"].sum(),
    "row_method": "50 ft gas transmission; 37.5 ft other included systems",
    "commodity_scope": "Include and Review retained; Exclude removed"
}])

statewide_path = (
    pipeline_folder / "pipeline_mixed_row_statewide_revised.csv"
)

county_path = (
    pipeline_folder / "pipeline_mixed_row_county_254_revised.csv"
)

statewide_revised.to_csv(statewide_path, index=False)

pipeline_county_final.to_csv(county_path, index=False)

# I read both files back to verify that the exported
# results match the validated in-memory calculations.

saved_statewide = pd.read_csv(statewide_path)

saved_county = pd.read_csv(
    county_path,
    dtype={"county_geoid": "string"}
)

assert np.isclose(
    saved_statewide["mixed_row_acres"].iloc[0],
    1497398.07,
    atol=0.01,
    rtol=0
)

assert len(saved_county) == 254

assert np.isclose(
    saved_county["mixed_row_acres"].sum(),
    saved_statewide["mixed_row_acres"].iloc[0],
    atol=0.01,
    rtol=0
)

print("Revised pipeline files saved successfully:")
print(statewide_path)
print(county_path)

print("\nFinal pipeline acreage:")
print(f"{saved_statewide['mixed_row_acres'].iloc[0]:,.2f}")

Revised pipeline files saved successfully:
G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\06_Energy_Footprint_Analysis\Pipelines\pipeline_mixed_row_statewide_revised.csv
G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\06_Energy_Footprint_Analysis\Pipelines\pipeline_mixed_row_county_254_revised.csv

Final pipeline acreage:
1,497,398.07


In [19]:
# NEW-NY
#Final pipeline updates based on Tony's feedback
#Based on Tony's comments, I reviewed the pipeline ROW assumptions.
#1. ROW widths
#My earlier mixed-width calculation used 50 feet for transmission pipelines and 37.5 feet for other pipelines.
#To make the estimate more conservative, I will use 25 feet for gathering and other included pipeline systems and keep 50 feet for transmission pipelines (system T).
#I will use the same pipeline mileage × ROW width method.

In [20]:
# NEW-NY
# I apply the revised ROW widths based on the data check.
# I keep my earlier calculations and save the new results separately.

pipeline_new = pipeline_final.copy()

# I use 50 feet for transmission pipelines and 25 feet for other systems.
pipeline_new["system_code"] = (
    pipeline_new["system_code"]
    .astype("string")
    .str.strip()
    .str.upper()
)

pipeline_new["row_width_ft"] = np.where(
    pipeline_new["system_code"].eq("T"),
    50.0,
    25.0
)

# I calculate the new ROW acreage using pipeline mileage.
pipeline_new["new_row_acres"] = (
    pipeline_new["miles"]
    * FT_PER_MILE
    * pipeline_new["row_width_ft"]
    / SQFT_PER_ACRE
)

# I compare the earlier and new acreage estimates by system.
pipeline_new_by_system = (
    pipeline_new.groupby(
        ["system_code", "row_width_ft"],
        as_index=False
    )
    .agg(
        pipeline_miles=("miles", "sum"),
        previous_acres=("revised_acres", "sum"),
        new_row_acres=("new_row_acres", "sum")
    )
)

display(pipeline_new_by_system.round(2))

# I calculate the new statewide pipeline footprint.
previous_total = pipeline_new["revised_acres"].sum()
new_total = pipeline_new["new_row_acres"].sum()

pipeline_new_statewide = pd.DataFrame([{
    "pipeline_miles": pipeline_new["miles"].sum(),
    "previous_acres": previous_total,
    "new_row_acres": new_total,
    "change_acres": new_total - previous_total,
    "row_method": "50 ft transmission; 25 ft other included systems"
}])

display(pipeline_new_statewide.round(2))

# I calculate the new acreage for each county.
pipeline_new_county = (
    pipeline_new.groupby("county_geoid", as_index=False)
    .agg(
        pipeline_miles=("miles", "sum"),
        new_row_acres=("new_row_acres", "sum")
    )
)

# I keep all Texas counties, including those with no pipelines.
pipeline_new_county_254 = (
    pipeline_county_final[
        ["county_geoid", "county_name"]
    ]
    .merge(
        pipeline_new_county,
        on="county_geoid",
        how="left",
        validate="one_to_one"
    )
)

pipeline_new_county_254[
    ["pipeline_miles", "new_row_acres"]
] = pipeline_new_county_254[
    ["pipeline_miles", "new_row_acres"]
].fillna(0)

# I check that the new results match across the statewide and county totals.
assert len(pipeline_new_county_254) == 254

assert np.isclose(
    pipeline_new_county_254["new_row_acres"].sum(),
    new_total,
    atol=0.01,
    rtol=0
)

assert np.isclose(
    pipeline_new_county_254["pipeline_miles"].sum(),
    pipeline_new["miles"].sum(),
    atol=0.01,
    rtol=0
)

assert new_total <= previous_total

# I save the new results without replacing the earlier files.
pipeline_new_by_system.to_csv(
    OUTPUT / "pipeline_new_row_by_system.csv",
    index=False
)

pipeline_new_statewide.to_csv(
    OUTPUT / "pipeline_new_row_statewide.csv",
    index=False
)

pipeline_new_county_254.to_csv(
    OUTPUT / "pipeline_new_row_county_254.csv",
    index=False
)

print("Previous pipeline footprint:", f"{previous_total:,.2f} acres")
print("New pipeline footprint:", f"{new_total:,.2f} acres")
print("Difference:", f"{new_total - previous_total:,.2f} acres")
print("New pipeline results saved.")

,system_code,row_width_ft,pipeline_miles,previous_acres,new_row_acres
0,G,25.0,170389.13,774496.07,516330.71
1,L,25.0,17175.12,78068.73,52045.82
2,O,25.0,26497.94,120445.17,80296.78
3,P,25.0,11586.30,52665.01,35110.00
4,Q,25.0,36617.55,166443.40,110962.27
5,T,50.0,50371.15,305279.70,305279.70


,pipeline_miles,previous_acres,new_row_acres,change_acres,row_method
0,312637.19,1497398.07,1100025.28,-397372.79,50 ft transmission; 25 ft other included systems


Previous pipeline footprint: 1,497,398.07 acres
New pipeline footprint: 1,100,025.28 acres
Difference: -397,372.79 acres
New pipeline results saved.


In [21]:
# NEW-NY: Spatial union preparation
# I check the geometry and coordinate system before buffering pipelines.
# I preserve my existing acreage calculations and output files.

from shapely.geometry import LineString, MultiLineString

print("Dataset type:", type(pipeline_new).__name__)
print("Pipeline records:", f"{len(pipeline_new):,}")
print("Pipeline miles:", f"{pipeline_new['miles'].sum():,.2f}")

# I confirm that my final eligible pipelines still have spatial geometry.
if not isinstance(pipeline_new, gpd.GeoDataFrame):
    raise TypeError("pipeline_new is not a GeoDataFrame.")

if pipeline_new.crs is None:
    raise ValueError("Pipeline geometry has no CRS.")

print("\nOriginal CRS:", pipeline_new.crs)
print("Geometry column:", pipeline_new.geometry.name)

# I inspect geometry types to identify anything unsuitable for line buffering.
print("\nGeometry types:")
display(pipeline_new.geometry.geom_type.value_counts())

# I check for missing, empty, or invalid geometries.
geometry_check = pd.DataFrame([{
    "records": len(pipeline_new),
    "missing_geometry": int(pipeline_new.geometry.isna().sum()),
    "empty_geometry": int(pipeline_new.geometry.is_empty.sum()),
    "invalid_geometry": int((~pipeline_new.geometry.is_valid).sum()),
    "missing_county": int(pipeline_new["county_geoid"].isna().sum()),
    "unique_counties": pipeline_new["county_geoid"].nunique(),
    "unique_source_records": pipeline_new["pipeline_record_id"].nunique()
}])

display(geometry_check)

# I check the ROW widths carried forward from my latest calculations.
display(
    pipeline_new.groupby(
        ["system_code", "row_width_ft"],
        dropna=False
    ).agg(
        records=("pipeline_record_id", "size"),
        miles=("miles", "sum"),
        gross_acres=("new_row_acres", "sum")
    ).reset_index().round(2)
)

# I verify that all geometries are valid line types before proceeding.
assert pipeline_new.geometry.notna().all()
assert not pipeline_new.geometry.is_empty.any()
assert pipeline_new.geometry.geom_type.isin(
    ["LineString", "MultiLineString"]
).all()

# I confirm that my revised gross acreage remains unchanged.
assert np.isclose(
    pipeline_new["new_row_acres"].sum(),
    1100025.28,
    atol=0.1,
    rtol=0
)

print("\nGeometry preparation checks completed.")

Dataset type: GeoDataFrame
Pipeline records: 445,438
Pipeline miles: 312,637.19

Original CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "NAD83 / Texas Centric Albers Equal Area", "base_crs": {"name": "NAD83", "datum": {"type": "GeodeticReferenceFrame", "name": "North American Datum 1983", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4269}}, "conversion": {"name": "Texas Centric Albers Equal Area", "method": {"name": "Albers Equal Area", "id": {"authority": "EPSG", "code": 9822}}, "parameters": [{"name": "Latitude of false origin", "value": 18, "unit": "degree", "id": {"authority": "EPSG", "code": 

MultiLineString    445438
Name: count, dtype: int64

,records,missing_geometry,empty_geometry,invalid_geometry,missing_county,unique_counties,unique_source_records
0,445438,0,0,0,0,253,435186


,system_code,row_width_ft,records,miles,gross_acres
0,G,25.0,339536,170389.13,516330.71
1,L,25.0,21518,17175.12,52045.82
2,O,25.0,6236,26497.94,80296.78
3,P,25.0,3551,11586.30,35110.00
4,Q,25.0,19957,36617.55,110962.27
5,T,50.0,54640,50371.15,305279.70



Geometry preparation checks completed.


In [22]:

# NEW-NY: I inspect county boundaries and split pipeline records.
# I want to avoid introducing artificial corridor areas at county borders.

# I check which county boundary datasets are already available.
for name in [
    "counties",
    "tx_counties",
    "counties_tx",
    "texas_counties",
    "county_gdf",
    "pipeline_counties"
]:
    if name in globals():
        obj = globals()[name]
        print(
            name,
            "| Type:", type(obj).__name__,
            "| Rows:", len(obj),
            "| CRS:", getattr(obj, "crs", None)
        )

# I count how many pipeline source records occur in multiple pieces.
piece_counts = pipeline_new.groupby(
    "pipeline_record_id"
).agg(
    pieces=("county_geoid", "size"),
    counties=("county_geoid", "nunique")
)

split_summary = pd.DataFrame([{
    "unique_source_records": len(piece_counts),
    "records_with_multiple_pieces": int(
        (piece_counts["pieces"] > 1).sum()
    ),
    "records_crossing_multiple_counties": int(
        (piece_counts["counties"] > 1).sum()
    ),
    "maximum_pieces_per_record": int(
        piece_counts["pieces"].max()
    )
}])

display(split_summary)

# I inspect the available county identifiers.
print("\nCounty GEOID examples:")
print(
    pipeline_new["county_geoid"]
    .astype("string")
    .drop_duplicates()
    .head(10)
    .tolist()
)

# I check whether the original pipeline geometry dataset is in memory.
print("\nPotential original pipeline datasets:")
for name, obj in list(globals().items()):
    if (
        isinstance(obj, gpd.GeoDataFrame)
        and "pipeline" in name.lower()
        and name != "pipeline_new"
    ):
        print(
            name,
            "| Rows:", len(obj),
            "| Geometry:", obj.geometry.geom_type.unique().tolist()
        )

,unique_source_records,records_with_multiple_pieces,records_crossing_multiple_counties,maximum_pieces_per_record
0,435186,9942,9942,4



County GEOID examples:
['48001', '48161', '48213', '48225', '48073', '48289', '48003', '48165', '48317', '48135']

Potential original pipeline datasets:
pipeline_classified | Rows: 446257 | Geometry: ['MultiLineString']
pipeline_mixed | Rows: 446257 | Geometry: ['MultiLineString']
pipeline_final | Rows: 445438 | Geometry: ['MultiLineString']


In [25]:

# NEW-NY: I inspect the original pipeline GeoPackage using pyogrio.
# I avoid loading the full dataset until I confirm its structure.

import pyogrio
import geopandas as gpd
from pathlib import Path

original_path = (
    ROOT / "03_Cleaned_Data" / "Texas_Pipelines_All_Statuses.gpkg"
)

assert original_path.exists(), f"File not found: {original_path}"

# I identify the available layers.
layers = pyogrio.list_layers(original_path)
print("Available layers:")
print(layers)

# I inspect the first layer's metadata.
layer_name = layers[0][0]

info = pyogrio.read_info(
    original_path,
    layer=layer_name
)

print("\nLayer:", layer_name)
print("Total features:", info["features"])
print("Geometry type:", info["geometry_type"])
print("CRS:", info["crs"])

print("\nAvailable columns:")
print(info["fields"].tolist())

# I read five records to inspect identifiers and attributes.
sample = gpd.read_file(
    original_path,
    layer=layer_name,
    rows=5,
    engine="pyogrio"
)

print("\nSample records:")
display(sample.drop(columns="geometry", errors="ignore"))

# I inspect the identifiers in my current eligible dataset.
print("\nCurrent pipeline identifiers:")
display(
    pipeline_new[
        ["pipeline_record_id", "system_code", "county_geoid"]
    ].head()
)

Available layers:
[['texas_pipelines' 'MultiLineString']]

Layer: texas_pipelines
Total features: 496305
Geometry type: MultiLineString
CRS: EPSG:4326

Available columns:
['TPMS_ID', 'OPER_NM', 'SYS_NM', 'DIAMETER', 'CMDTY_DESC', 'STATUS_CD', 'SYSTYPE', 'COUNTY', 'ALBERS_MIL', 'source_county', 'geometry_miles', 'status_description', 'system_description']

Sample records:


,TPMS_ID,OPER_NM,SYS_NM,DIAMETER,CMDTY_DESC,STATUS_CD,SYSTYPE,COUNTY,ALBERS_MIL,source_county,geometry_miles,status_description,system_description
0,6720370,"TRINITY GAS STORAGE, LLC",BETHEL PIPELINE SYSTEM,16.00,NATURAL GAS,I,G,001,0.125550,001,0.125499,In Service,Gas Gathering
1,1985934,"JAY MANAGEMENT COMPANY, LLC",CAYUGA GATHERING LINE,2.38,NATURAL GAS,B,G,001,1.200550,001,1.204075,Abandoned,Gas Gathering
2,6921813,"CALPINE TEXAS PIPELINE, L.P.",FREESTONE,12.75,NATURAL GAS,I,T,001,1.572640,001,1.571611,In Service,Gas Transmission
3,6921814,"CALPINE TEXAS PIPELINE, L.P.",FREESTONE,12.75,NATURAL GAS,I,T,001,0.048124,001,0.048069,In Service,Gas Transmission
4,6921815,"CALPINE TEXAS PIPELINE, L.P.",FREESTONE,20.00,NATURAL GAS,I,T,001,5.483962,001,5.475655,In Service,Gas Transmission



Current pipeline identifiers:


,pipeline_record_id,system_code,county_geoid
0,0,G,48001
3,2,T,48001
4,3,T,48001
5,4,T,48001
6,4,T,48161


In [27]:
# NEW-NY: I validate the existing source IDs before spatial union.
# I reuse my final eligibility decisions and ROW widths without recalculating them.

# I confirm that each source pipeline has one consistent ROW width.
row_check = pipeline_new.groupby("TPMS_ID")["row_width_ft"].nunique()

print("Source IDs with conflicting ROW widths:", (row_check > 1).sum())
print("Missing TPMS_ID values:", pipeline_new["TPMS_ID"].isna().sum())

# I create a compact lookup of eligible source IDs and their final ROW widths.
eligible_pipeline_lookup = (
    pipeline_new[["TPMS_ID", "row_width_ft"]]
    .drop_duplicates()
    .copy()
)

print("Eligible source IDs:", f"{eligible_pipeline_lookup['TPMS_ID'].nunique():,}")

assert pipeline_new["TPMS_ID"].notna().all()
assert (row_check == 1).all()
assert eligible_pipeline_lookup["TPMS_ID"].is_unique

# I confirm that the existing gross footprint remains unchanged.
assert np.isclose(
    pipeline_new["new_row_acres"].sum(),
    1100025.28,
    atol=0.1,
    rtol=0
)

print("Source identifier and ROW checks passed.")

Source IDs with conflicting ROW widths: 0
Missing TPMS_ID values: 0
Eligible source IDs: 428,599
Source identifier and ROW checks passed.


In [28]:
# NEW-NY: I load the original centerlines for my eligible pipelines.
# I reuse the final eligibility decisions and ROW widths from pipeline_new.

# I read the original pipeline geometry without changing the source file.
original_pipelines = gpd.read_file(
    original_path,
    layer="texas_pipelines",
    columns=["TPMS_ID"],
    engine="pyogrio"
)

print("Original pipeline features:", f"{len(original_pipelines):,}")

# I retain only the source IDs included in my final analysis.
eligible_ids = set(eligible_pipeline_lookup["TPMS_ID"])

original_eligible = original_pipelines.loc[
    original_pipelines["TPMS_ID"].isin(eligible_ids)
].copy()

# I attach the previously validated ROW widths.
original_eligible = original_eligible.merge(
    eligible_pipeline_lookup,
    on="TPMS_ID",
    how="left",
    validate="many_to_one"
)

# I convert to the Texas equal-area projected CRS.
original_eligible = original_eligible.to_crs("EPSG:3083")

# I check the geometry and ROW information.
print("Eligible original features:", f"{len(original_eligible):,}")
print("Unique eligible source IDs:", f"{original_eligible['TPMS_ID'].nunique():,}")
print("Missing ROW widths:", original_eligible["row_width_ft"].isna().sum())
print("Invalid geometries:", (~original_eligible.geometry.is_valid).sum())

# I confirm that every eligible source ID was recovered.
missing_ids = eligible_ids - set(original_eligible["TPMS_ID"])

print("Eligible IDs missing from original file:", len(missing_ids))

assert len(missing_ids) == 0
assert original_eligible["row_width_ft"].notna().all()
assert original_eligible.geometry.notna().all()
assert not original_eligible.geometry.is_empty.any()

print("\nOriginal eligible pipeline geometries are ready.")

Original pipeline features: 496,305
Eligible original features: 435,186
Unique eligible source IDs: 428,599
Missing ROW widths: 0
Invalid geometries: 0
Eligible IDs missing from original file: 0

Original eligible pipeline geometries are ready.


In [31]:
# NEW-NY: I load the existing Texas county boundaries.
# I use the file in my organized raw data folder.

county_path = (
    ROOT / "01_Raw_Data" / "Texas_Boundaries"
    / "tl_2026_us_county" / "tl_2026_us_county.shp"
)

texas_counties = gpd.read_file(
    county_path,
    where="STATEFP = '48'",
    engine="pyogrio"
).to_crs("EPSG:3083")

# I confirm that all Texas counties and Reeves County are present.
assert len(texas_counties) == 254
assert texas_counties["GEOID"].nunique() == 254

reeves = texas_counties.loc[
    texas_counties["GEOID"] == "48389"
].copy()

assert len(reeves) == 1

print("Texas counties:", len(texas_counties))
print("CRS:", texas_counties.crs.to_epsg())
print("Reeves County:", reeves["NAME"].iloc[0])
print("Reeves boundary ready.")

Texas counties: 254
CRS: 3083
Reeves County: Reeves
Reeves boundary ready.


In [32]:
# NEW-NY: I calculate the unique pipeline corridor footprint in Reeves County.
# I buffer original centerlines before clipping to avoid artificial county endpoints.
# I merge overlapping ROW corridors so each geographic area is counted once.

from shapely import union_all
from shapely.geometry import box

METERS_PER_FOOT = 0.3048
SQM_PER_ACRE = 4046.8564224

# I select the Reeves County boundary.
reeves_geom = reeves.geometry.iloc[0]

# I use a 25-foot search radius because this is half my maximum ROW width.
max_buffer_m = 25.0 * METERS_PER_FOOT

# I find all original pipeline features near Reeves County.
# I include neighboring features whose ROW could extend into the county.
search_area = reeves_geom.buffer(max_buffer_m)

candidate_idx = list(
    original_eligible.sindex.query(
        search_area,
        predicate="intersects"
    )
)

reeves_pipelines = original_eligible.iloc[candidate_idx].copy()

print("Candidate pipeline features:", f"{len(reeves_pipelines):,}")

# I buffer each centerline by half its assigned ROW width.
# I use flat ends to avoid adding artificial ROW area at pipeline endpoints.
buffered_corridors = [
    geom.buffer(
        (width / 2.0) * METERS_PER_FOOT,
        cap_style="flat",
        join_style="round"
    )
    for geom, width in zip(
        reeves_pipelines.geometry,
        reeves_pipelines["row_width_ft"]
    )
]

# I merge overlapping corridors into a single geographic footprint.
reeves_union = union_all(buffered_corridors)

# I clip the merged footprint to the Reeves County boundary.
reeves_unique_geom = reeves_union.intersection(reeves_geom)

# I calculate the unique modeled corridor acreage.
reeves_unique_acres = reeves_unique_geom.area / SQM_PER_ACRE

# I retrieve the gross acreage from my finalized county results.
reeves_gross_acres = pipeline_new_county_254.loc[
    pipeline_new_county_254["county_geoid"] == "48389",
    "new_row_acres"
].iloc[0]

# I compare the existing gross estimate with the spatial union result.
reeves_comparison = pd.DataFrame([{
    "county": "Reeves",
    "gross_row_acres": reeves_gross_acres,
    "unique_union_acres": reeves_unique_acres,
    "difference_acres": reeves_gross_acres - reeves_unique_acres,
    "difference_pct": (
        (reeves_gross_acres - reeves_unique_acres)
        / reeves_gross_acres * 100
        if reeves_gross_acres > 0 else np.nan
    )
}])

display(reeves_comparison.round(2))

print("\nReeves County spatial union test completed.")

Candidate pipeline features: 11,809


,county,gross_row_acres,unique_union_acres,difference_acres,difference_pct
0,Reeves,29499.09,26124.68,3374.41,11.44



Reeves County spatial union test completed.


In [33]:
# NEW-NY: I calculate unique pipeline ROW acreage for all 254 Texas counties.
# I reuse my final eligible centerlines, ROW widths, and county boundaries.
# I merge overlapping buffers and clip the resulting footprint to each county.
# I save checkpoints so I can resume without repeating completed counties.

from shapely import union_all
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import gc
import time

METERS_PER_FOOT = 0.3048
SQM_PER_ACRE = 4046.8564224

# I save the spatial union results separately from my previous estimates.
UNION_OUT = OUTPUT / "Pipeline_Unique_ROW"
UNION_OUT.mkdir(parents=True, exist_ok=True)

checkpoint_path = UNION_OUT / "pipeline_union_checkpoint.csv"
county_output_path = UNION_OUT / "pipeline_unique_row_county_254.csv"
state_output_path = UNION_OUT / "pipeline_unique_row_statewide.csv"

# I retain the maximum possible buffer radius for boundary searches.
max_buffer_m = (
    original_eligible["row_width_ft"].max()
    / 2 * METERS_PER_FOOT
)

# I use the spatial index already associated with my original centerlines.
pipeline_sindex = original_eligible.sindex

# I restore completed results if a previous run was interrupted.
if checkpoint_path.exists():
    completed = pd.read_csv(
        checkpoint_path,
        dtype={"county_geoid": "string"}
    )
    completed["county_geoid"] = completed["county_geoid"].str.zfill(5)
    results = completed.to_dict("records")
else:
    results = []

done = {r["county_geoid"] for r in results}

start_time = time.time()

for _, county in texas_counties.iterrows():

    geoid = str(county["GEOID"]).zfill(5)

    if geoid in done:
        continue

    county_geom = county.geometry

    # I search beyond the county boundary to capture nearby ROW corridors.
    search_geom = county_geom.buffer(max_buffer_m)

    candidate_idx = pipeline_sindex.query(
        search_geom,
        predicate="intersects"
    )

    candidates = original_eligible.iloc[candidate_idx]

    # I buffer each eligible centerline using half its assigned ROW width.
    buffers = [
        geom.buffer(
            float(width) / 2 * METERS_PER_FOOT,
            cap_style="flat",
            join_style="round"
        )
        for geom, width in zip(
            candidates.geometry,
            candidates["row_width_ft"]
        )
    ]

    # I merge overlapping corridors before clipping to the county.
    if buffers:
        merged = union_all(buffers)
        county_union = merged.intersection(county_geom)
        unique_acres = county_union.area / SQM_PER_ACRE
    else:
        unique_acres = 0.0

    results.append({
        "county_geoid": geoid,
        "county_name": county["NAME"],
        "candidate_features": len(candidates),
        "unique_row_acres": unique_acres
    })

    done.add(geoid)

    # I save progress regularly to avoid losing completed calculations.
    if len(results) % 10 == 0:
        pd.DataFrame(results).to_csv(
            checkpoint_path,
            index=False
        )
        print(
            f"Completed {len(results)}/254 counties | "
            f"Latest: {county['NAME']}"
        )

    del buffers
    gc.collect()

# I save the complete county results.
union_counties = pd.DataFrame(results)

assert len(union_counties) == 254
assert union_counties["county_geoid"].nunique() == 254

# I compare unique acreage with my existing gross ROW estimates.
union_counties = union_counties.merge(
    pipeline_new_county_254[
        ["county_geoid", "pipeline_miles", "new_row_acres"]
    ],
    on="county_geoid",
    how="left",
    validate="one_to_one"
)

union_counties["difference_acres"] = (
    union_counties["new_row_acres"]
    - union_counties["unique_row_acres"]
)

union_counties["difference_pct"] = np.where(
    union_counties["new_row_acres"] > 0,
    union_counties["difference_acres"]
    / union_counties["new_row_acres"] * 100,
    np.nan
)

union_counties = union_counties.sort_values(
    "county_geoid"
).reset_index(drop=True)

union_counties.to_csv(
    county_output_path,
    index=False
)

# I calculate statewide totals from the nonoverlapping county boundaries.
gross_total = union_counties["new_row_acres"].sum()
unique_total = union_counties["unique_row_acres"].sum()

union_statewide = pd.DataFrame([{
    "pipeline_miles": union_counties["pipeline_miles"].sum(),
    "gross_row_acres": gross_total,
    "unique_row_acres": unique_total,
    "difference_acres": gross_total - unique_total,
    "difference_pct": (
        (gross_total - unique_total) / gross_total * 100
    ),
    "row_method": "50 ft transmission; 25 ft other included systems"
}])

union_statewide.to_csv(
    state_output_path,
    index=False
)

# I confirm that my original gross acreage remains consistent.
assert np.isclose(
    gross_total,
    pipeline_new["new_row_acres"].sum(),
    atol=0.01,
    rtol=0
)

print("\nSTATEWIDE PIPELINE ROW COMPARISON")
display(union_statewide.round(2))

print("\nCompleted counties:", len(union_counties))
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))
print("Results saved to:", UNION_OUT)

Completed 10/254 counties | Latest: Chambers
Completed 20/254 counties | Latest: Bell
Completed 30/254 counties | Latest: Potter
Completed 40/254 counties | Latest: Rockwall
Completed 50/254 counties | Latest: Calhoun
Completed 60/254 counties | Latest: Jim Wells
Completed 70/254 counties | Latest: Grayson
Completed 80/254 counties | Latest: Caldwell
Completed 90/254 counties | Latest: Cherokee
Completed 100/254 counties | Latest: Callahan


GEOSException: bad allocation

In [34]:
# NEW-NY: I resume my statewide pipeline union without repeating completed counties.
# I process buffers in smaller batches to reduce peak memory usage.
# I preserve my previous checkpoint and final output files.

from shapely import union_all
from shapely.errors import GEOSException
import pandas as pd
import numpy as np
import gc
import time

BATCH_SIZE = 500

# I restore my completed county calculations.
completed = pd.read_csv(
    checkpoint_path,
    dtype={"county_geoid": "string"}
)

completed["county_geoid"] = (
    completed["county_geoid"].str.zfill(5)
)

results = completed.to_dict("records")
done = set(completed["county_geoid"])

print("Counties already completed:", len(done))
print("Counties remaining:", 254 - len(done))

# I merge geometries incrementally instead of passing every buffer at once.
def batched_union(geometries, batch_size=500):

    partial_unions = []

    for start in range(0, len(geometries), batch_size):
        batch = geometries[start:start + batch_size]
        partial_unions.append(union_all(batch))

    # I combine the smaller union results into the final geometry.
    return union_all(partial_unions)


start_time = time.time()

for _, county in texas_counties.iterrows():

    geoid = str(county["GEOID"]).zfill(5)

    if geoid in done:
        continue

    print(f"Processing {county['NAME']} ({geoid})")

    try:
        county_geom = county.geometry

        # I include centerlines whose ROW extends into the county.
        search_geom = county_geom.buffer(max_buffer_m)

        candidate_idx = pipeline_sindex.query(
            search_geom,
            predicate="intersects"
        )

        candidates = original_eligible.iloc[candidate_idx]

        # I create the same ROW buffers used in my earlier calculations.
        buffers = [
            geom.buffer(
                float(width) / 2 * METERS_PER_FOOT,
                cap_style="flat",
                join_style="round"
            )
            for geom, width in zip(
                candidates.geometry,
                candidates["row_width_ft"]
            )
        ]

        if buffers:
            merged = batched_union(
                buffers,
                batch_size=BATCH_SIZE
            )

            county_union = merged.intersection(county_geom)
            unique_acres = county_union.area / SQM_PER_ACRE
        else:
            unique_acres = 0.0

        results.append({
            "county_geoid": geoid,
            "county_name": county["NAME"],
            "candidate_features": len(candidates),
            "unique_row_acres": unique_acres
        })

        done.add(geoid)

        # I save every successful county to avoid repeating work.
        pd.DataFrame(results).to_csv(
            checkpoint_path,
            index=False
        )

        del buffers
        gc.collect()

    except (GEOSException, MemoryError) as error:
        print(f"FAILED: {county['NAME']} ({geoid})")
        print("Error:", error)

        # I preserve successful results and continue with other counties.
        pd.DataFrame(results).to_csv(
            checkpoint_path,
            index=False
        )
        gc.collect()
        continue

print("\nCompleted counties:", len(done))
print("Remaining counties:", 254 - len(done))
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))

# I only create final statewide results after all counties succeed.
if len(done) == 254:

    union_counties = pd.DataFrame(results)

    union_counties = union_counties.merge(
        pipeline_new_county_254[
            ["county_geoid", "pipeline_miles", "new_row_acres"]
        ],
        on="county_geoid",
        how="left",
        validate="one_to_one"
    )

    union_counties["difference_acres"] = (
        union_counties["new_row_acres"]
        - union_counties["unique_row_acres"]
    )

    union_counties["difference_pct"] = np.where(
        union_counties["new_row_acres"] > 0,
        union_counties["difference_acres"]
        / union_counties["new_row_acres"] * 100,
        np.nan
    )

    union_counties = union_counties.sort_values("county_geoid")

    union_counties.to_csv(
        county_output_path,
        index=False
    )

    gross_total = union_counties["new_row_acres"].sum()
    unique_total = union_counties["unique_row_acres"].sum()

    union_statewide = pd.DataFrame([{
        "pipeline_miles": union_counties["pipeline_miles"].sum(),
        "gross_row_acres": gross_total,
        "unique_row_acres": unique_total,
        "difference_acres": gross_total - unique_total,
        "difference_pct": (
            (gross_total - unique_total) / gross_total * 100
        ),
        "row_method": "50 ft transmission; 25 ft other included systems"
    }])

    assert np.isclose(
        gross_total,
        pipeline_new["new_row_acres"].sum(),
        atol=0.01,
        rtol=0
    )

    union_statewide.to_csv(
        state_output_path,
        index=False
    )

    print("\nSTATEWIDE RESULTS")
    display(union_statewide.round(2))

else:
    print("\nSome counties remain incomplete.")
    print("I will not calculate a final statewide footprint yet.")

Counties already completed: 100
Counties remaining: 154
Processing Moore (48341)
Processing Maverick (48323)
Processing Cooke (48097)
Processing Angelina (48005)
Processing Nueces (48355)
Processing Aransas (48007)
Processing San Patricio (48409)
Processing Scurry (48415)
Processing Reeves (48389)
Processing Hood (48221)
Processing Tom Green (48451)
Processing Irion (48235)
Processing Wilson (48493)
Processing Kendall (48259)
Processing Guadalupe (48187)
Processing Medina (48325)
Processing Bexar (48029)
Processing Atascosa (48013)
Processing Bandera (48019)
Processing Comal (48091)
Processing Hidalgo (48215)
Processing Midland (48329)
Processing Martin (48317)
Processing Palo Pinto (48363)
Processing Camp (48063)
Processing Morris (48343)
Processing Titus (48449)
Processing Hopkins (48223)
Processing Archer (48009)
Processing Wichita (48485)
Processing Clay (48077)
Processing Zapata (48505)
Processing Webb (48479)
Processing Presidio (48377)
Processing Baylor (48023)
Processing San Au

,pipeline_miles,gross_row_acres,unique_row_acres,difference_acres,difference_pct,row_method
0,312637.19,1100025.28,984144.19,115881.09,10.53,50 ft transmission; 25 ft other included systems


In [35]:
# NEW-NY: I validate my completed statewide pipeline union results.
# I use the saved county results without repeating any spatial calculations.

# I confirm that all Texas counties are represented exactly once.
assert len(union_counties) == 254
assert union_counties["county_geoid"].nunique() == 254

# I compare the saved Reeves County result with my earlier test.
reeves_final = union_counties.loc[
    union_counties["county_geoid"] == "48389"
]

print("REEVES COUNTY VALIDATION")
display(reeves_final.round(2))

assert np.isclose(
    reeves_final["unique_row_acres"].iloc[0],
    reeves_unique_acres,
    atol=0.01,
    rtol=0
)

# I identify counties where unique acreage exceeds the gross estimate.
higher_unique = union_counties.loc[
    union_counties["difference_acres"] < -0.01
].copy()

print("\nCounties with unique acreage greater than gross acreage:")
print(len(higher_unique))

display(
    higher_unique[
        ["county_name", "new_row_acres",
         "unique_row_acres", "difference_acres"]
    ].sort_values("difference_acres").head(15).round(2)
)

# I identify counties with the largest acreage reductions.
print("\nCounties with the largest reductions:")
display(
    union_counties[
        ["county_name", "new_row_acres",
         "unique_row_acres", "difference_acres",
         "difference_pct"]
    ].sort_values(
        "difference_acres", ascending=False
    ).head(15).round(2)
)

# I confirm that county acreage reconciles with the statewide result.
assert np.isclose(
    union_counties["unique_row_acres"].sum(),
    union_statewide["unique_row_acres"].iloc[0],
    atol=0.01,
    rtol=0
)

print("\nStatewide reconciliation passed.")

REEVES COUNTY VALIDATION


,county_geoid,county_name,candidate_features,unique_row_acres,pipeline_miles,new_row_acres,difference_acres,difference_pct
108,48389,Reeves,11809,26124.68,8968.77,29499.09,3374.41,11.44



Counties with unique acreage greater than gross acreage:
0


,county_name,new_row_acres,unique_row_acres,difference_acres



Counties with the largest reductions:


,county_name,new_row_acres,unique_row_acres,difference_acres,difference_pct
11,Harris,28120.28,16729.56,11390.72,40.51
13,Brazoria,20363.74,13738.57,6625.17,32.53
23,Jefferson,15836.03,10861.71,4974.32,31.41
108,Reeves,29499.09,26124.68,3374.41,11.44
9,Chambers,8401.93,5672.75,2729.19,32.48
104,Nueces,11817.16,9285.17,2531.99,21.43
201,Dimmit,9039.77,6941.10,2098.66,23.22
10,Galveston,6438.67,4340.64,2098.03,32.58
121,Midland,22950.54,20866.58,2083.96,9.08
239,Karnes,10851.39,8840.82,2010.56,18.53



Statewide reconciliation passed.
